In [2]:
import pandas as pd
from pathlib import Path

# Project folders
DATA_DIR = Path(".")
CLEAN_DIR = DATA_DIR / "cleaned"
OUTPUT_DIR = DATA_DIR / "outputs"

# Load cleaned CWEEDS dataset
cweeds = pd.read_csv(
    CLEAN_DIR / "clean_cweeds_4cities.csv",
    parse_dates=["timestamp"],
    low_memory=False
)

print("CWEEDS loaded successfully")
print("Shape:", cweeds.shape)
print("Date range:", cweeds["timestamp"].min(), "to", cweeds["timestamp"].max())
print("\nCities:")
print(cweeds["city"].value_counts())

CWEEDS loaded successfully
Shape: (701280, 64)
Date range: 1998-01-01 00:00:00 to 2017-12-31 23:00:00

Cities:
city
Calgary         175320
Edmonton        175320
Lethbridge      175320
Medicine Hat    175320
Name: count, dtype: int64


In [3]:
powerbi_solar_dashboard = (
    cweeds
    .assign(
        Year=cweeds["timestamp"].dt.year,
        Month=cweeds["timestamp"].dt.month,
        Hour=cweeds["timestamp"].dt.hour
    )
    .groupby(
        ["city", "Year", "Month", "Hour"],
        as_index=False
    )
    .agg(
        Mean_GHI_Wh_m2=("ghi_wh_m2", "mean"),
        Mean_DNI_Wh_m2=("dni_wh_m2", "mean"),
        Mean_DHI_Wh_m2=("dhi_wh_m2", "mean"),
        Mean_Temperature_C=("dry_bulb_temp_c", "mean"),
        Mean_Dew_Point_C=("dew_point_temp_c", "mean"),
        Mean_Wind_Speed_ms=("wind_speed_ms", "mean"),
        Mean_Station_Pressure_Pa=("station_pressure_pa", "mean")
    )
    .round(2)
)

display(powerbi_solar_dashboard.head(20))

print("Power BI dashboard rows:", f"{len(powerbi_solar_dashboard):,}")

,city,Year,Month,Hour,Mean_GHI_Wh_m2,Mean_DNI_Wh_m2,Mean_DHI_Wh_m2,Mean_Temperature_C,Mean_Dew_Point_C,Mean_Wind_Speed_ms,Mean_Station_Pressure_Pa
0,Calgary,1998,1,0,0.00,0.00,0.00,-14.52,-19.36,2.54,88377.74
1,Calgary,1998,1,1,0.00,0.00,0.00,-14.87,-19.33,2.93,88376.45
2,Calgary,1998,1,2,0.00,0.00,0.00,-14.86,-19.56,2.84,88386.13
3,Calgary,1998,1,3,0.00,0.00,0.00,-14.87,-19.49,2.59,88381.29
4,Calgary,1998,1,4,0.00,0.00,0.00,-14.55,-19.25,2.69,88370.65
5,Calgary,1998,1,5,0.00,0.00,0.00,-14.72,-19.28,2.92,88370.65
6,Calgary,1998,1,6,0.00,0.00,0.00,-14.77,-19.12,2.75,88370.97
7,Calgary,1998,1,7,0.00,0.00,0.00,-14.36,-18.77,2.70,88386.45
8,Calgary,1998,1,8,3.96,1.89,2.01,-14.47,-18.58,2.95,88398.39
9,Calgary,1998,1,9,52.50,106.41,37.92,-14.02,-18.43,2.95,88424.19


Power BI dashboard rows: 23,040


In [4]:
OUTPUT_DIR.mkdir(exist_ok=True)

output_path = OUTPUT_DIR / "powerbi_solar_dashboard.csv"

powerbi_solar_dashboard.to_csv(
    output_path,
    index=False
)

print("Saved:", output_path)

Saved: outputs\powerbi_solar_dashboard.csv
